# Boosting Algorithm Experiments — Alzheimer's Diagnosis Classification

**Target**: `DIAGNOSIS` (1.0 = CN, 2.0 = MCI, 3.0 = Dementia)  
**Features**: 6 ordinal CDR domains + 1 categorical GENOTYPE  
**Evaluation**: Stratified 5-fold CV with macro F1-score  

Models tested:
1. XGBoost
2. LightGBM
3. CatBoost
4. AdaBoost
5. Gradient Boosting (sklearn)
6. HistGradientBoosting (sklearn)

---
## 1. Setup & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, accuracy_score, make_scorer
)
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
N_FOLDS = 5
TARGET = 'DIAGNOSIS'
LABEL_MAP = {1.0: 'CN', 2.0: 'MCI', 3.0: 'Dementia'}

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('Setup complete.')

---
## 2. Data Loading

In [ ]:
DATA_DIR = '../data'

DATASETS = {
    'ordinal_imbalanced': f'{DATA_DIR}/balanced_diagnosis_ordinal.csv',
    'ordinal_smotenc':    f'{DATA_DIR}/balanced_diagnosis_ordinal_smotenc.csv',
    'ordinal_ctgan':      f'{DATA_DIR}/balanced_diagnosis_ordinal_ctgan.csv',
}

datasets = {}
for name, path in DATASETS.items():
    df = pd.read_csv(path)
    datasets[name] = df
    print(f'{name:25s} -> {df.shape[0]:>6,} rows, {df.shape[1]} cols | class dist: {dict(df[TARGET].value_counts().sort_index())}')

datasets['ordinal_imbalanced'].head()

---
## 3. Feature Engineering Helpers

In [ ]:
CDR_COLS = ['CDMEMORY', 'CDORIENT', 'CDJUDGE', 'CDCOMMUN', 'CDHOME', 'CDCARE']
CAT_COL = 'GENOTYPE'
GENOTYPE_ORDER = ['2/2', '2/3', '2/4', '3/3', '3/4', '4/4']
CLASS_NAMES = ['CN', 'MCI', 'Dementia']

TARGET_LABEL_ENC = LabelEncoder()
TARGET_LABEL_ENC.fit([1.0, 2.0, 3.0])  # 1.0 -> 0, 2.0 -> 1, 3.0 -> 2


def prepare_features(df, encode_genotype='ordinal'):
    """Prepare X, y from the ordinal dataframe.

    encode_genotype:
        'ordinal'  -> integer-encode GENOTYPE (suitable for tree models)
        'native'   -> keep as-is (for CatBoost native categorical support)

    y is remapped to 0-indexed [0, 1, 2] for XGBoost compatibility.
    """
    X = df.drop(columns=[TARGET]).copy()
    y = TARGET_LABEL_ENC.transform(df[TARGET].values)

    if encode_genotype == 'ordinal':
        enc = OrdinalEncoder(categories=[GENOTYPE_ORDER])
        X[CAT_COL] = enc.fit_transform(X[[CAT_COL]]).astype(int)
    elif encode_genotype == 'native':
        pass  # keep string column for CatBoost

    return X, y


print(f'Feature columns: {CDR_COLS + [CAT_COL]}')
print(f'Target: {TARGET} -> remapped to {{0: CN, 1: MCI, 2: Dementia}}')

---
## 4. Cross-Validation Engine

In [ ]:
def run_cv(model, X, y, n_folds=N_FOLDS, model_name='Model'):
    """Run stratified k-fold CV and return a results dict."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    scoring = {
        'accuracy':  'accuracy',
        'f1_macro':  make_scorer(f1_score, average='macro'),
        'f1_weighted': make_scorer(f1_score, average='weighted'),
    }

    cv_results = cross_validate(
        model, X, y,
        cv=skf,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1,
    )

    summary = {
        'model':              model_name,
        'train_acc':          cv_results['train_accuracy'].mean(),
        'val_acc':            cv_results['test_accuracy'].mean(),
        'val_acc_std':        cv_results['test_accuracy'].std(),
        'train_f1_macro':     cv_results['train_f1_macro'].mean(),
        'val_f1_macro':       cv_results['test_f1_macro'].mean(),
        'val_f1_macro_std':   cv_results['test_f1_macro'].std(),
        'val_f1_weighted':    cv_results['test_f1_weighted'].mean(),
        'fit_time':           cv_results['fit_time'].mean(),
    }

    print(f"  {model_name:30s} | val F1-macro = {summary['val_f1_macro']:.4f} ± {summary['val_f1_macro_std']:.4f} "
          f"| val Acc = {summary['val_acc']:.4f} | fit {summary['fit_time']:.2f}s")

    return summary


print('CV engine ready.')

---
## 5. Model Definitions

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.tree import DecisionTreeClassifier


def get_models(random_state=RANDOM_STATE):
    """Return a dict of {name: (model, encode_mode)}.

    encode_mode tells prepare_features how to handle GENOTYPE.
    """
    models = {
        'XGBoost': (
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                use_label_encoder=False,
                eval_metric='mlogloss',
                random_state=random_state,
                n_jobs=-1,
                verbosity=0,
            ),
            'ordinal',
        ),
        'LightGBM': (
            LGBMClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=random_state,
                n_jobs=-1,
                verbose=-1,
            ),
            'ordinal',
        ),
        'CatBoost': (
            CatBoostClassifier(
                iterations=300,
                depth=6,
                learning_rate=0.1,
                l2_leaf_reg=3.0,
                random_state=random_state,
                verbose=0,
            ),
            'ordinal',
        ),
        'AdaBoost': (
            AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=3),
                n_estimators=300,
                learning_rate=0.1,
                random_state=random_state,
                algorithm='SAMME',
            ),
            'ordinal',
        ),
        'GradientBoosting': (
            GradientBoostingClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                random_state=random_state,
            ),
            'ordinal',
        ),
        'HistGradientBoosting': (
            HistGradientBoostingClassifier(
                max_iter=300,
                max_depth=6,
                learning_rate=0.1,
                random_state=random_state,
            ),
            'ordinal',
        ),
    }
    return models


print(f'Defined {len(get_models())} boosting models.')

---
## 6. Run Experiments on All Datasets

In [ ]:
all_results = []

for ds_name, df in datasets.items():
    print(f'\n{"=" * 70}')
    print(f'Dataset: {ds_name}  ({len(df):,} rows)')
    print(f'{"=" * 70}')

    models = get_models()

    for model_name, (model, encode_mode) in models.items():
        X, y = prepare_features(df, encode_genotype=encode_mode)
        result = run_cv(model, X, y, model_name=model_name)
        result['dataset'] = ds_name
        all_results.append(result)

results_df = pd.DataFrame(all_results)
print('\nAll experiments complete.')

---
## 7. Results Summary Table

In [ ]:
summary_cols = ['dataset', 'model', 'train_f1_macro', 'val_f1_macro', 'val_f1_macro_std',
                'val_acc', 'val_acc_std', 'val_f1_weighted', 'fit_time']

display_df = results_df[summary_cols].copy()
display_df = display_df.sort_values(['dataset', 'val_f1_macro'], ascending=[True, False])

display_df.style.format({
    'train_f1_macro':   '{:.4f}',
    'val_f1_macro':     '{:.4f}',
    'val_f1_macro_std': '{:.4f}',
    'val_acc':          '{:.4f}',
    'val_acc_std':      '{:.4f}',
    'val_f1_weighted':  '{:.4f}',
    'fit_time':         '{:.2f}s',
}).background_gradient(subset=['val_f1_macro'], cmap='YlGn')

---
## 8. Visualisation — Model Comparison

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(7 * len(datasets), 5), sharey=True)
if len(datasets) == 1:
    axes = [axes]

for ax, ds_name in zip(axes, datasets):
    subset = results_df[results_df['dataset'] == ds_name].sort_values('val_f1_macro', ascending=True)
    colors = sns.color_palette('viridis', n_colors=len(subset))
    bars = ax.barh(subset['model'], subset['val_f1_macro'], xerr=subset['val_f1_macro_std'],
                   color=colors, edgecolor='black', linewidth=0.5, capsize=3)
    ax.set_xlabel('Validation F1-macro')
    ax.set_title(ds_name, fontweight='bold')
    ax.set_xlim(0, 1.0)
    for bar, val in zip(bars, subset['val_f1_macro']):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)

plt.suptitle('Boosting Models — Validation F1-macro by Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Overfitting Analysis (Train vs Val)

In [ ]:
results_df['overfit_gap'] = results_df['train_f1_macro'] - results_df['val_f1_macro']

fig, ax = plt.subplots(figsize=(10, 5))
pivot = results_df.pivot(index='model', columns='dataset', values='overfit_gap')
pivot.plot(kind='bar', ax=ax, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Train F1-macro − Val F1-macro')
ax.set_title('Overfitting Gap per Model & Dataset', fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(title='Dataset', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 10. Confusion Matrices for Best Model per Dataset

In [ ]:
from sklearn.model_selection import cross_val_predict

fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 5))
if len(datasets) == 1:
    axes = [axes]

for ax, ds_name in zip(axes, datasets):
    best_row = results_df[results_df['dataset'] == ds_name].sort_values('val_f1_macro', ascending=False).iloc[0]
    best_model_name = best_row['model']

    models = get_models()
    model, encode_mode = models[best_model_name]
    X, y = prepare_features(datasets[ds_name], encode_genotype=encode_mode)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    y_pred = cross_val_predict(model, X, y, cv=skf, n_jobs=-1)

    cm = confusion_matrix(y, y_pred, labels=[0, 1, 2])
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{ds_name}\nBest: {best_model_name} (F1={best_row["val_f1_macro"]:.4f})', fontweight='bold')

plt.suptitle('Confusion Matrices — Best Boosting Model per Dataset', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

---
## 11. Per-Class Classification Reports

In [ ]:
for ds_name in datasets:
    best_row = results_df[results_df['dataset'] == ds_name].sort_values('val_f1_macro', ascending=False).iloc[0]
    best_model_name = best_row['model']

    models = get_models()
    model, encode_mode = models[best_model_name]
    X, y = prepare_features(datasets[ds_name], encode_genotype=encode_mode)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    y_pred = cross_val_predict(model, X, y, cv=skf, n_jobs=-1)

    print(f'\n{"=" * 60}')
    print(f'Dataset: {ds_name}  |  Best model: {best_model_name}')
    print(f'{"=" * 60}')
    print(classification_report(y, y_pred, target_names=CLASS_NAMES, digits=4))

---
## 12. Feature Importance (Best Model)

In [ ]:
from sklearn.inspection import permutation_importance

chosen_ds = 'ordinal_smotenc'
best_row = results_df[results_df['dataset'] == chosen_ds].sort_values('val_f1_macro', ascending=False).iloc[0]
best_model_name = best_row['model']

models = get_models()
model, encode_mode = models[best_model_name]
X, y = prepare_features(datasets[chosen_ds], encode_genotype=encode_mode)

model.fit(X, y)

perm_imp = permutation_importance(model, X, y,
                                  scoring=make_scorer(f1_score, average='macro'),
                                  n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)

imp_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm_imp.importances_mean,
    'importance_std':  perm_imp.importances_std,
}).sort_values('importance_mean', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_df['feature'], imp_df['importance_mean'], xerr=imp_df['importance_std'],
        color=sns.color_palette('viridis', n_colors=len(imp_df)), edgecolor='black', linewidth=0.5, capsize=3)
ax.set_xlabel('Permutation Importance (F1-macro drop)')
ax.set_title(f'Feature Importance — {best_model_name} on {chosen_ds}', fontweight='bold')
plt.tight_layout()
plt.show()

print(imp_df.sort_values('importance_mean', ascending=False).to_string(index=False))

---
## 13. Export Results

In [ ]:
output_path = '../data/boosting_cv_results.csv'
results_df.to_csv(output_path, index=False)
print(f'Results saved to {output_path}')
results_df.head()

---
## 14. Binary Classification — CN vs Cognitively Impaired (MCI + Dementia)

The 3-class model struggles at the CN–MCI boundary. Clinically, the key question is often:
**"Is this patient cognitively normal or not?"**

Below we collapse MCI and Dementia into a single **Impaired** class and re-run all boosting models.

In [ ]:
BINARY_CLASS_NAMES = ['CN', 'Impaired']


def prepare_features_binary(df, encode_genotype='ordinal'):
    """Prepare X, y with binary target: CN (0) vs Impaired (1).

    MCI (2.0) and Dementia (3.0) are merged into Impaired.
    """
    X = df.drop(columns=[TARGET]).copy()
    y = (df[TARGET].values != 1.0).astype(int)  # CN=0, MCI+Dementia=1

    if encode_genotype == 'ordinal':
        enc = OrdinalEncoder(categories=[GENOTYPE_ORDER])
        X[CAT_COL] = enc.fit_transform(X[[CAT_COL]]).astype(int)

    return X, y


for ds_name, df in datasets.items():
    _, y_bin = prepare_features_binary(df)
    cn = (y_bin == 0).sum()
    imp = (y_bin == 1).sum()
    print(f'{ds_name:25s} -> CN: {cn:,}  Impaired: {imp:,}  (ratio {imp/cn:.2f}:1)')

### 14a. Run Binary Experiments

In [ ]:
binary_results = []

for ds_name, df in datasets.items():
    print(f'\n{"=" * 70}')
    print(f'[BINARY] Dataset: {ds_name}  ({len(df):,} rows)')
    print(f'{"=" * 70}')

    models = get_models()

    for model_name, (model, encode_mode) in models.items():
        X, y = prepare_features_binary(df, encode_genotype=encode_mode)
        result = run_cv(model, X, y, model_name=model_name)
        result['dataset'] = ds_name
        binary_results.append(result)

binary_results_df = pd.DataFrame(binary_results)
print('\nAll binary experiments complete.')

### 14b. Binary Results Table

In [ ]:
bin_display = binary_results_df[summary_cols].copy()
bin_display = bin_display.sort_values(['dataset', 'val_f1_macro'], ascending=[True, False])

bin_display.style.format({
    'train_f1_macro':   '{:.4f}',
    'val_f1_macro':     '{:.4f}',
    'val_f1_macro_std': '{:.4f}',
    'val_acc':          '{:.4f}',
    'val_acc_std':      '{:.4f}',
    'val_f1_weighted':  '{:.4f}',
    'fit_time':         '{:.2f}s',
}).background_gradient(subset=['val_f1_macro'], cmap='YlGn')

### 14c. Binary vs 3-Class Comparison

In [ ]:
results_df['task'] = '3-class'
binary_results_df['task'] = 'binary'
combined_df = pd.concat([results_df, binary_results_df], ignore_index=True)

fig, axes = plt.subplots(1, len(datasets), figsize=(7 * len(datasets), 5), sharey=True)
if len(datasets) == 1:
    axes = [axes]

for ax, ds_name in zip(axes, datasets):
    subset = combined_df[combined_df['dataset'] == ds_name]
    pivot = subset.pivot(index='model', columns='task', values='val_f1_macro')
    pivot = pivot.sort_values('binary', ascending=True)
    pivot.plot(kind='barh', ax=ax, edgecolor='black', linewidth=0.5, width=0.7)
    ax.set_xlabel('Validation F1-macro')
    ax.set_title(ds_name, fontweight='bold')
    ax.set_xlim(0, 1.05)
    ax.legend(title='Task')

plt.suptitle('Binary (CN vs Impaired) vs 3-Class — F1-macro Comparison',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 14d. Binary Confusion Matrices & Classification Reports

In [ ]:
from sklearn.model_selection import cross_val_predict as _cvp

fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 4))
if len(datasets) == 1:
    axes = [axes]

for ax, ds_name in zip(axes, datasets):
    best_row = binary_results_df[binary_results_df['dataset'] == ds_name] \
                   .sort_values('val_f1_macro', ascending=False).iloc[0]
    best_model_name = best_row['model']

    models = get_models()
    model, encode_mode = models[best_model_name]
    X, y = prepare_features_binary(datasets[ds_name], encode_genotype=encode_mode)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    y_pred = _cvp(model, X, y, cv=skf, n_jobs=-1)

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=BINARY_CLASS_NAMES)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{ds_name}\nBest: {best_model_name} (F1={best_row["val_f1_macro"]:.4f})', fontweight='bold')

plt.suptitle('Binary Confusion Matrices — Best Boosting Model per Dataset',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print('\n' + '=' * 70)
print('PER-CLASS REPORTS')
print('=' * 70)

for ds_name in datasets:
    best_row = binary_results_df[binary_results_df['dataset'] == ds_name] \
                   .sort_values('val_f1_macro', ascending=False).iloc[0]
    best_model_name = best_row['model']

    models = get_models()
    model, encode_mode = models[best_model_name]
    X, y = prepare_features_binary(datasets[ds_name], encode_genotype=encode_mode)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    y_pred = _cvp(model, X, y, cv=skf, n_jobs=-1)

    print(f'\n--- {ds_name}  |  Best: {best_model_name} ---')
    print(classification_report(y, y_pred, target_names=BINARY_CLASS_NAMES, digits=4))

### 14e. Export All Results (3-class + Binary)

In [ ]:
output_path = '../data/boosting_cv_results_all.csv'
combined_df.to_csv(output_path, index=False)
print(f'Combined results (3-class + binary) saved to {output_path}')
print(f'Total experiments: {len(combined_df)}')
combined_df.groupby('task')[['val_f1_macro', 'val_acc']].agg(['mean', 'max']).round(4)

---
## 15. Hierarchical Cascade Classifier — CN vs (MCI vs Dementia)

**Architecture**: Stage 1 (HistGradientBoosting) classifies CN vs Not-CN. For samples predicted Not-CN, Stage 2 distinguishes MCI vs Dementia.

Each stage solves a simpler, more focused sub-problem. Wrapped as a single sklearn estimator for honest cross-validation.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone


class HierarchicalClassifier(BaseEstimator, ClassifierMixin):
    """Two-stage cascade: Stage 1 splits CN vs Not-CN,
    Stage 2 splits MCI vs Dementia on the Not-CN subset.

    Expects 3-class y: 0=CN, 1=MCI, 2=Dementia.
    """

    def __init__(self, stage1=None, stage2=None):
        self.stage1 = stage1
        self.stage2 = stage2

    def fit(self, X, y):
        self.classes_ = np.array([0, 1, 2])

        X_arr = np.asarray(X)
        y_arr = np.asarray(y)

        # Stage 1: CN (0) vs Not-CN (1)
        y_bin = (y_arr != 0).astype(int)
        self.stage1_ = clone(self.stage1)
        self.stage1_.fit(X_arr, y_bin)

        # Stage 2: MCI (0) vs Dementia (1), trained only on impaired samples
        mask = y_arr != 0
        y_s2 = (y_arr[mask] - 1).astype(int)  # MCI(1)->0, Dementia(2)->1
        self.stage2_ = clone(self.stage2)
        self.stage2_.fit(X_arr[mask], y_s2)

        return self

    def predict(self, X):
        X_arr = np.asarray(X)
        preds = np.zeros(len(X_arr), dtype=int)

        s1_preds = self.stage1_.predict(X_arr)

        cn_mask = s1_preds == 0
        impaired_mask = ~cn_mask

        preds[cn_mask] = 0  # CN

        if impaired_mask.any():
            s2_preds = self.stage2_.predict(X_arr[impaired_mask])
            preds[impaired_mask] = s2_preds + 1  # 0->1(MCI), 1->2(Dementia)

        return preds


print('HierarchicalClassifier defined.')

### 15a. Stage 2 Candidate Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC


def get_stage1(random_state=RANDOM_STATE):
    """Fixed Stage 1: HistGradientBoosting for CN vs Not-CN."""
    return HistGradientBoostingClassifier(
        max_iter=300,
        max_depth=6,
        learning_rate=0.1,
        random_state=random_state,
    )


def get_stage2_candidates(random_state=RANDOM_STATE):
    """Return candidate models for Stage 2 (MCI vs Dementia)."""
    return {
        'S2-HistGB': HistGradientBoostingClassifier(
            max_iter=300, max_depth=6, learning_rate=0.1,
            random_state=random_state,
        ),
        'S2-XGBoost': XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            use_label_encoder=False, eval_metric='logloss',
            random_state=random_state, n_jobs=-1, verbosity=0,
        ),
        'S2-LightGBM': LGBMClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            random_state=random_state, n_jobs=-1, verbose=-1,
        ),
        'S2-LogisticRegression': LogisticRegression(
            max_iter=1000, solver='lbfgs',
            random_state=random_state,
        ),
        'S2-SVM-RBF': SVC(
            kernel='rbf', C=1.0,
            random_state=random_state,
        ),
        'S2-SVM-Linear': SVC(
            kernel='linear', C=1.0,
            random_state=random_state,
        ),
    }


print(f'Stage 1: HistGradientBoosting (fixed)')
print(f'Stage 2 candidates: {list(get_stage2_candidates().keys())}')

### 15b. Run Hierarchical Experiments

In [ ]:
hierarchical_results = []

for ds_name, df in datasets.items():
    print(f'\n{"=" * 70}')
    print(f'[HIERARCHICAL] Dataset: {ds_name}  ({len(df):,} rows)')
    print(f'{"=" * 70}')

    X, y = prepare_features(df, encode_genotype='ordinal')
    stage2_candidates = get_stage2_candidates()

    for s2_name, s2_model in stage2_candidates.items():
        pipeline = HierarchicalClassifier(
            stage1=get_stage1(),
            stage2=s2_model,
        )
        result = run_cv(pipeline, X, y, model_name=s2_name)
        result['dataset'] = ds_name
        hierarchical_results.append(result)

hierarchical_df = pd.DataFrame(hierarchical_results)
print('\nAll hierarchical experiments complete.')

### 15c. Hierarchical Results Table

In [ ]:
hier_display = hierarchical_df[summary_cols].copy()
hier_display = hier_display.sort_values(['dataset', 'val_f1_macro'], ascending=[True, False])

hier_display.style.format({
    'train_f1_macro':   '{:.4f}',
    'val_f1_macro':     '{:.4f}',
    'val_f1_macro_std': '{:.4f}',
    'val_acc':          '{:.4f}',
    'val_acc_std':      '{:.4f}',
    'val_f1_weighted':  '{:.4f}',
    'fit_time':         '{:.2f}s',
}).background_gradient(subset=['val_f1_macro'], cmap='YlGn')

### 15d. Comparison — Hierarchical vs Flat 3-Class vs Flat Binary

In [ ]:
hierarchical_df['task'] = 'hierarchical'

all_tasks_df = pd.concat([combined_df, hierarchical_df], ignore_index=True)

best_per_task = all_tasks_df.groupby(['dataset', 'task'])['val_f1_macro'].max().reset_index()
best_pivot = best_per_task.pivot(index='dataset', columns='task', values='val_f1_macro')

fig, ax = plt.subplots(figsize=(10, 5))
best_pivot.plot(kind='bar', ax=ax, edgecolor='black', linewidth=0.5, width=0.75)
ax.set_ylabel('Best Validation F1-macro')
ax.set_title('Best Model F1-macro: Flat 3-Class vs Binary vs Hierarchical', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(title='Task', bbox_to_anchor=(1.02, 1), loc='upper left')
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', fontsize=8, padding=2)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

print('\nBest F1-macro per task & dataset:')
print(best_pivot.round(4).to_string())

### 15e. Confusion Matrices & Classification Reports (Hierarchical)

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 5))
if len(datasets) == 1:
    axes = [axes]

for ax, ds_name in zip(axes, datasets):
    best_row = hierarchical_df[hierarchical_df['dataset'] == ds_name] \
                   .sort_values('val_f1_macro', ascending=False).iloc[0]
    best_s2_name = best_row['model']

    s2_candidates = get_stage2_candidates()
    pipeline = HierarchicalClassifier(
        stage1=get_stage1(),
        stage2=s2_candidates[best_s2_name],
    )

    X, y = prepare_features(datasets[ds_name], encode_genotype='ordinal')
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    y_pred = cross_val_predict(pipeline, X, y, cv=skf, n_jobs=-1)

    cm = confusion_matrix(y, y_pred, labels=[0, 1, 2])
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{ds_name}\nBest S2: {best_s2_name} (F1={best_row["val_f1_macro"]:.4f})', fontweight='bold')

plt.suptitle('Hierarchical Classifier — Confusion Matrices', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print('\n' + '=' * 70)
print('PER-CLASS REPORTS (HIERARCHICAL)')
print('=' * 70)

for ds_name in datasets:
    best_row = hierarchical_df[hierarchical_df['dataset'] == ds_name] \
                   .sort_values('val_f1_macro', ascending=False).iloc[0]
    best_s2_name = best_row['model']

    s2_candidates = get_stage2_candidates()
    pipeline = HierarchicalClassifier(
        stage1=get_stage1(),
        stage2=s2_candidates[best_s2_name],
    )

    X, y = prepare_features(datasets[ds_name], encode_genotype='ordinal')
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    y_pred = cross_val_predict(pipeline, X, y, cv=skf, n_jobs=-1)

    print(f'\n--- {ds_name}  |  Stage 2: {best_s2_name} ---')
    print(classification_report(y, y_pred, target_names=CLASS_NAMES, digits=4))

### 15f. Export All Results (3-class + Binary + Hierarchical)

In [ ]:
output_path = '../data/boosting_cv_results_all.csv'
all_tasks_df.to_csv(output_path, index=False)
print(f'All results saved to {output_path}')
print(f'Total experiments: {len(all_tasks_df)}')
print()
all_tasks_df.groupby('task')[['val_f1_macro', 'val_acc']].agg(['mean', 'max']).round(4)